# R50 · RSB_FULL_06_ldm_extra1361_fromscratch

**Domanda sperimentale:** effetto di `RSB_FULL_06_ldm_extra1361_fromscratch` su ResNet-50, con confronto validation-only e tre seed indipendenti. Questo notebook non può leggere il test locked.


## 1–5 · Identità e regime

- Experiment ID logico: `resnet50__RSB_FULL_06_ldm_extra1361_fromscratch`
- Architettura: `resnet50` (ResNet-50)
- Dataset variant: `RSB_FULL_06_ldm_extra1361_fromscratch`
- Regime: `full_available`
- Generatore: `06_ldm_extra1361_fromscratch`


In [ ]:
from pathlib import Path
import json, os, sys

def find_project_root(start=Path.cwd()):
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "configs/classifier_experiment_matrix.json").is_file():
            return candidate
    raise FileNotFoundError("MammoDiffusion project root not found")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "notebooks/utility"))
import classifier_experiment_runner as runner
import classifier_dataset_builder as datasets

ARCHITECTURE = 'resnet50'
DATASET_VARIANT_ID = 'RSB_FULL_06_ldm_extra1361_fromscratch'
EXPERIMENT_ID = 'resnet50__RSB_FULL_06_ldm_extra1361_fromscratch'
MODE = "auto"  # plan | auto | train | validate | metrics-only
RUN_SEEDS = [17, 42, 73]
ALLOW_RETRAIN = False
ALLOW_OVERWRITE_VERIFIED = False
TINY_SMOKE = os.environ.get("MAMMO_CLASSIFIER_TINY") == "1"
DATASET_STATUS = 'READY'
DATASET_BLOCKER = None
assert MODE != "locked-test"
print(EXPERIMENT_ID, MODE, RUN_SEEDS, "tiny=", TINY_SMOKE)


## 6–9 · Provenance, conteggi, firme e split pazienti

La cella seguente risolve i file canonici, firma il manifest e carica esclusivamente la validation reale. Le varianti bloccate restano documentate e non avviano training.


In [ ]:
registry = runner.load_dataset_variant_registry(PROJECT_ROOT)
variant = next(v for v in registry["variants"] if v["dataset_variant_id"] == DATASET_VARIANT_ID)
if DATASET_STATUS == "BLOCKED":
    dataset_summary = {"status": DATASET_STATUS, "blocker": DATASET_BLOCKER}
else:
    train_rows, validation_rows, dataset_manifest = datasets.build_training_and_validation_rows(PROJECT_ROOT, variant)
    dataset_summary = {
        "status": DATASET_STATUS, "counts": dataset_manifest["counts"],
        "train_samples": len(train_rows), "validation_samples": len(validation_rows),
        "dataset_signature": dataset_manifest["signature"],
        "validation_signature": dataset_manifest["validation_signature"],
        "validation_sources": sorted({row["source"] for row in validation_rows}),
    }
print(json.dumps(dataset_summary, indent=1))


## 10–13 · Protocollo, seed, piano auto e resume

Il protocollo è unico per architettura. `auto` riusa un checkpoint verificato, altrimenti addestra, poi esegue validation e metriche. Ogni seed usa directory e checkpoint distinti.


In [ ]:
policy = runner.load_training_protocols(PROJECT_ROOT)["policies"][ARCHITECTURE]
plans = [runner.plan(PROJECT_ROOT, ARCHITECTURE, DATASET_VARIANT_ID, seed) for seed in RUN_SEEDS]
print(json.dumps({"policy": policy, "plans": plans}, indent=1))


## 14–18 · Curve, validation, metriche seed, ensemble e soglia

Questa è l’unica cella operativa. Notebook e scheduler chiamano la stessa funzione condivisa. L’ensemble è la media delle probabilità dei seed 17/42/73 e la soglia è scelta sull’ensemble validation.


In [ ]:
if DATASET_STATUS == "BLOCKED":
    run_results = [{"status": "BLOCKED", "reason": DATASET_BLOCKER}]
else:
    run_results = runner.execute_configuration(
        PROJECT_ROOT, ARCHITECTURE, DATASET_VARIANT_ID,
        mode=MODE, run_seeds=RUN_SEEDS, tiny=TINY_SMOKE,
    )
print(json.dumps(run_results, indent=1, default=str))


## 19 · Consumi

I consumi per seed sono registrati nella registry di sostenibilità durante le esecuzioni reali; la modalità tiny è sempre identificata come smoke e non entra nel consuntivo scientifico.


In [ ]:
policy_name = f"{ARCHITECTURE}_standard"
ensemble_path = (PROJECT_ROOT / "results/classifiers_matrix" / ARCHITECTURE /
                 DATASET_VARIANT_ID / policy_name / "ensemble_validation_manifest.json")
print("ensemble:", ensemble_path, "exists=", ensemble_path.is_file())
for seed in RUN_SEEDS:
    run_dir = runner.resolve_job(PROJECT_ROOT, ARCHITECTURE, DATASET_VARIANT_ID, seed)["run_dir"]
    print(seed, run_dir, sorted(p.name for p in run_dir.glob("*.json")) if run_dir.exists() else [])


## 20 · Riepilogo e output

Output canonici: `experiments/classifiers_matrix/<arch>/<variant>/<policy>/seed_<seed>/` e `results/classifiers_matrix/<arch>/<variant>/<policy>/ensemble_validation_manifest.json`. Il test locked non è importato né accessibile da questo notebook.
